# Stage 9: Final Model Evaluation

Consolidate saved results from Stages 5-8. No models are fitted here and test results do not alter prior selection. Metrics refer to citywide hourly `ridership_count`, with Entry and Exit retained as separate series.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == 'Notebooks':
    project_root = project_root.parent
data_dir = project_root / 'DataProcessed'
notebook_dir = project_root / 'Notebooks'
stage_notebooks = {stage: json.loads((notebook_dir / filename).read_text()) for stage, filename in {5: '07_Baseline_Forecasting.ipynb', 6: '08_Feature_Diagnostics_Ablation.ipynb', 7: '09_Advanced_Forecasting.ipynb', 8: '10_Error_Analysis.ipynb'}.items()}
assert set(stage_notebooks) == {5, 6, 7, 8}
stage5_metrics = pd.read_csv(data_dir / 'baseline_forecasting_metrics.csv')
stage6_metrics = pd.read_csv(data_dir / 'feature_ablation_metrics.csv')
stage7_metrics = pd.read_csv(data_dir / 'advanced_forecasting_metrics.csv')
stage8_errors = pd.read_csv(data_dir / 'stage8_test_error_breakdown.csv')
stage8_weather = pd.read_csv(data_dir / 'stage8_weather_value_test.csv')
stage5_predictions = pd.read_csv(data_dir / 'baseline_forecasting_predictions.csv', parse_dates=['time'])
stage6_predictions = pd.read_csv(data_dir / 'feature_ablation_predictions.csv', parse_dates=['time'])
stage7_predictions = pd.read_csv(data_dir / 'advanced_forecasting_predictions.csv', parse_dates=['time'])
prepared = {split: pd.read_csv(data_dir / f'metro_weather_{split}_prepared.csv', parse_dates=['time']) for split in ['train', 'validation', 'test']}
for split, frame in prepared.items():
    assert not frame.duplicated(['time', 'direction']).any()
    assert frame[['time', 'direction', 'ridership_count']].notna().all().all()
    assert set(frame.direction) == {'Entry', 'Exit'}
assert prepared['train'].time.max() < prepared['validation'].time.min() < prepared['test'].time.min()
assert len(prepared['validation']) == len(prepared['test']) == 810
print('Loaded saved metrics/predictions for Stages 5-7, Stage 8 error summaries, and chronological prepared splits.')

Loaded saved metrics/predictions for Stages 5-7, Stage 8 error summaries, and chronological prepared splits.


## Consolidated Metrics and Selection

Validation winners below are reconstructed from saved validation MAE only. Test metrics are reported for comparison and were not used to select models. Stage 8 contributes subgroup diagnostics, not a new fitted model.

In [2]:
def canonical_metrics(frame, stage, sample_col):
    output = frame.rename(columns={sample_col: 'n'}).copy()
    output['stage'] = stage
    output['dimension'] = 'overall'
    output['group'] = 'all'
    if 'MAPE_pct' not in output:
        output['MAPE_pct'] = np.nan
    return output[['stage', 'split', 'model', 'dimension', 'group', 'n', 'MAE', 'RMSE', 'MAPE_pct']]

consolidated = pd.concat([
    canonical_metrics(stage5_metrics, 'Stage 5 baseline', 'n_predictions'),
    canonical_metrics(stage6_metrics, 'Stage 6 linear ablation', 'n'),
    canonical_metrics(stage7_metrics, 'Stage 7 advanced', 'n'),
    stage8_errors.assign(stage='Stage 8 test diagnostics', split='test', MAPE_pct=np.nan)[['stage', 'split', 'model', 'dimension', 'group', 'n', 'MAE', 'RMSE', 'MAPE_pct']]
], ignore_index=True)
selection_specs = [
    ('Stage 5 baseline', stage5_metrics, 'n_predictions'),
    ('Stage 6 linear ablation', stage6_metrics, 'n'),
    ('Stage 7 advanced', stage7_metrics, 'n')
]
selection_rows = []
for stage_name, table, sample_col in selection_specs:
    validation = table.loc[table.split.eq('validation')].sort_values(['MAE', 'RMSE'])
    winner = validation.iloc[0]
    test_row = table.loc[table.split.eq('test') & table.model.eq(winner.model)].iloc[0]
    selection_rows.append({
        'stage': stage_name, 'validation_selected_model': winner.model,
        'validation_n': int(winner[sample_col]), 'validation_MAE': winner.MAE, 'validation_RMSE': winner.RMSE, 'validation_MAPE_pct': winner.MAPE_pct,
        'test_n': int(test_row[sample_col]), 'test_MAE': test_row.MAE, 'test_RMSE': test_row.RMSE, 'test_MAPE_pct': test_row.MAPE_pct,
        'test_minus_validation_MAE': test_row.MAE-winner.MAE, 'test_minus_validation_RMSE': test_row.RMSE-winner.RMSE,
        'selection_rule': 'minimum validation MAE; test reserved for final reporting'
    })
selection_summary = pd.DataFrame(selection_rows)
assert selection_summary.loc[selection_summary.stage.eq('Stage 7 advanced'), 'validation_selected_model'].iloc[0] == 'RandomForest_B_plus_weather'

weather_rows = []
def add_weather_pair(source_stage, family, table, name_a, name_b, sample_col):
    for split in ['validation', 'test']:
        a = table.loc[table.split.eq(split) & table.model.eq(name_a)].iloc[0]
        b = table.loc[table.split.eq(split) & table.model.eq(name_b)].iloc[0]
        weather_rows.append({'stage': source_stage, 'family': family, 'split': split, 'n': int(a[sample_col]), 'MAE_no_weather': a.MAE, 'MAE_with_weather': b.MAE, 'MAE_change_B_minus_A': b.MAE-a.MAE, 'RMSE_no_weather': a.RMSE, 'RMSE_with_weather': b.RMSE, 'RMSE_change_B_minus_A': b.RMSE-a.RMSE})
add_weather_pair('Stage 6', 'LinearRegression', stage6_metrics, 'A_temporal_history', 'B_plus_weather', 'n')
add_weather_pair('Stage 7', 'RandomForest', stage7_metrics, 'RandomForest_A_temporal_history', 'RandomForest_B_plus_weather', 'n')
add_weather_pair('Stage 7', 'HistGradientBoosting', stage7_metrics, 'HistGradientBoosting_A_temporal_history', 'HistGradientBoosting_B_plus_weather', 'n')
weather_value = pd.DataFrame(weather_rows)
rainfall_findings = stage8_weather.rename(columns={'rainfall_group': 'group', 'MAE_no_weather': 'MAE_A', 'MAE_with_weather': 'MAE_B', 'MAE_change_B_minus_A': 'MAE_delta', 'RMSE_no_weather': 'RMSE_A', 'RMSE_with_weather': 'RMSE_B', 'RMSE_change_B_minus_A': 'RMSE_delta'}).copy()
rainfall_findings['scope'] = 'Stage 8 test RF stratification; observed rainfall group, paired RF predictions'

metrics_path = data_dir / 'stage9_consolidated_metrics.csv'
selection_path = data_dir / 'stage9_validation_selection_summary.csv'
weather_path = data_dir / 'stage9_weather_contribution_summary.csv'
regional_path = data_dir / 'stage9_regional_status_summary.csv'
consolidated.to_csv(metrics_path, index=False)
selection_summary.to_csv(selection_path, index=False)
weather_value.to_csv(weather_path, index=False)
regional_status = pd.DataFrame([
    {'topic': 'Existing descriptive geography', 'finding': 'OBSERVATION.md reports station and metro-line cumulative demand patterns (LINE02 highest, LINE03 second; exits exceed entries by line). This is descriptive line/station analysis, not regional weather forecasting.', 'status': 'completed descriptive finding', 'source': 'Documents/OBSERVATION.md'},
    {'topic': 'Geographic regional forecasting', 'finding': 'No region-level predictions, weather joins, or regional metrics were found in saved outputs. The weather roadmap lists station-to-region mapping and regional weather integration as planned extensions.', 'status': 'not completed; do not infer regional model performance', 'source': 'Documents/2. Weather Data collection & Integration roadmap.md; DataProcessed artifact inventory'}
])
regional_status.to_csv(regional_path, index=False)
print('Validation-selected models and their held-out test metrics:')
display(selection_summary)
print('Weather A/B effects (negative B-A is lower error):')
display(weather_value)
print('Stage 8 rainfall-stratified RF results:')
display(rainfall_findings[['group', 'n', 'MAE_A', 'MAE_B', 'MAE_delta', 'RMSE_A', 'RMSE_B', 'RMSE_delta', 'sample_note']])
print('Saved summary tables:', metrics_path, selection_path, weather_path, regional_path)

Validation-selected models and their held-out test metrics:


,stage,validation_selected_model,validation_n,validation_MAE,validation_RMSE,validation_MAPE_pct,test_n,test_MAE,test_RMSE,test_MAPE_pct,test_minus_validation_MAE,test_minus_validation_RMSE,selection_rule
0,Stage 5 baseline,linear_regression,810,15309.669837,19916.905574,872.771082,810,15808.969128,21902.348972,923.808366,499.299291,1985.443398,minimum validation MAE; test reserved for fina...
1,Stage 6 linear ablation,A_temporal_history,810,14930.636897,19941.441927,981.605613,810,15870.571792,21703.634552,884.926160,939.934896,1762.192625,minimum validation MAE; test reserved for fina...
2,Stage 7 advanced,RandomForest_B_plus_weather,810,5778.637351,10106.005026,4.735941,810,4794.907553,7804.432450,4.897345,-983.729797,-2301.572576,minimum validation MAE; test reserved for fina...


Weather A/B effects (negative B-A is lower error):


,stage,family,split,n,MAE_no_weather,MAE_with_weather,MAE_change_B_minus_A,RMSE_no_weather,RMSE_with_weather,RMSE_change_B_minus_A
0,Stage 6,LinearRegression,validation,810,14930.636897,15151.819070,221.182173,19941.441927,19858.557215,-82.884712
1,Stage 6,LinearRegression,test,810,15870.571792,15861.764066,-8.807726,21703.634552,21788.450642,84.816091
2,Stage 7,RandomForest,validation,810,5849.316393,5778.637351,-70.679042,10107.701423,10106.005026,-1.696397
3,Stage 7,RandomForest,test,810,4449.933389,4794.907553,344.974165,7127.348612,7804.432450,677.083837
4,Stage 7,HistGradientBoosting,validation,810,6271.642787,6751.119987,479.477200,9789.255688,10088.203082,298.947394
5,Stage 7,HistGradientBoosting,test,810,4702.928558,5112.383317,409.454758,6965.695302,7569.095621,603.400320


Stage 8 rainfall-stratified RF results:


,group,n,MAE_A,MAE_B,MAE_delta,RMSE_A,RMSE_B,RMSE_delta,sample_note
0,All test rows,810,4449.933389,4794.907553,344.974165,7127.348612,7804.432450,677.083837,ok
1,No Rain,778,4451.842475,4815.748300,363.905825,7095.248498,7818.106928,722.858430,ok
2,Light Rain,30,4539.425845,4309.062138,-230.363707,8098.466523,7631.998601,-466.467922,ok
3,Moderate Rain,2,2364.911952,3975.538253,1610.626300,2570.010634,4211.118277,1641.107644,sparse (n<10)
4,Heavy Rain,0,NaN,NaN,NaN,NaN,NaN,NaN,empty


Saved summary tables: /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/stage9_consolidated_metrics.csv /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/stage9_validation_selection_summary.csv /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/stage9_weather_contribution_summary.csv /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/stage9_regional_status_summary.csv


## Consistency Audit

In [4]:
def verify_saved_metrics(metric_frame, prediction_frame, split_col='split', model_col='model'):
    for split in ['validation', 'test']:
        prepared_frame = prepared[split]
        expected = prepared_frame.set_index(['time', 'direction'])['ridership_count'].sort_index()
        split_predictions = prediction_frame.loc[prediction_frame[split_col].eq(split)]
        for model_name, group in split_predictions.groupby(model_col):
            indexed = group.set_index(['time', 'direction']).sort_index()
            assert indexed.index.equals(expected.index)
            assert np.array_equal(indexed['actual'].to_numpy(), expected.to_numpy())
            assert indexed['prediction'].notna().all()
            error = indexed['prediction'].to_numpy() - expected.to_numpy()
            saved = metric_frame.loc[metric_frame[split_col].eq(split) & metric_frame[model_col].eq(model_name)].iloc[0]
            assert np.isclose(np.mean(np.abs(error)), saved.MAE)
            assert np.isclose(np.sqrt(np.mean(error ** 2)), saved.RMSE)
            nonzero = expected.to_numpy() != 0
            expected_mape = np.mean(np.abs(error[nonzero] / expected.to_numpy()[nonzero])) * 100
            assert np.isclose(expected_mape, saved.MAPE_pct)

verify_saved_metrics(stage5_metrics, stage5_predictions)
verify_saved_metrics(stage6_metrics, stage6_predictions)
verify_saved_metrics(stage7_metrics, stage7_predictions)
assert stage5_metrics.loc[stage5_metrics.model.eq('seasonal_naive_24h')].sort_values('split')[['MAE', 'RMSE']].reset_index(drop=True).equals(stage7_metrics.loc[stage7_metrics.model.eq('SeasonalNaive_24h')].sort_values('split')[['MAE', 'RMSE']].reset_index(drop=True))
for split in ['validation', 'test']:
    for model_a, model_b in [('A_temporal_history', 'LinearRegression_A_temporal_history'), ('B_plus_weather', 'LinearRegression_B_temporal_weather')]:
        a = stage6_metrics.loc[stage6_metrics.split.eq(split) & stage6_metrics.model.eq(model_a)].iloc[0]
        b = stage7_metrics.loc[stage7_metrics.split.eq(split) & stage7_metrics.model.eq(model_b)].iloc[0]
        assert np.isclose(a.MAE, b.MAE) and np.isclose(a.RMSE, b.RMSE)
assert stage8_errors.n.ge(0).all() and stage8_errors.loc[stage8_errors.n.eq(0), ['MAE', 'RMSE']].isna().all().all()
assert stage8_weather.loc[stage8_weather.rainfall_group.eq('All test rows'), 'n'].iloc[0] == 810
regional_model_artifacts = [path for path in data_dir.glob('*region*') if path.name != regional_path.name]
assert not regional_model_artifacts
for path in [metrics_path, selection_path, weather_path, regional_path]:
    assert path.exists() and path.stat().st_size > 0
print('PASS: all saved Stage 5-7 MAE/RMSE/MAPE independently reproduced; Stage 5 seasonal-naive and Stage 6 reused linear metrics match Stage 7; Stage 8 subgroup counts/empty groups are valid.')
print('Regional result status: no regional forecasting artifacts; only the cited descriptive station/line EDA exists.')
print('No model fitting, test-driven selection, or source-output overwrite occurred in Stage 9.')

PASS: all saved Stage 5-7 MAE/RMSE/MAPE independently reproduced; Stage 5 seasonal-naive and Stage 6 reused linear metrics match Stage 7; Stage 8 subgroup counts/empty groups are valid.
Regional result status: no regional forecasting artifacts; only the cited descriptive station/line EDA exists.
No model fitting, test-driven selection, or source-output overwrite occurred in Stage 9.


## Final Research Conclusions

- **Best model:** Stage 7's validation-selected `RandomForest_B_plus_weather` (validation MAE/RMSE 5,778.6/10,106.0); test MAE/RMSE 4,794.9/7,804.4. Test performance did not determine selection.
- **Benchmarks:** Test RF A (no weather) scored 4,449.9/7,127.3, seasonal naive 22,585.7/42,670.2, and Stage 5 linear regression 15,809.0/21,902.3. Weather RF was selected on validation but did not lead on test.
- **Rainfall/weather contribution:** Incremental weather value is inconsistent and not established: RF weather worsened overall test MAE by 345.0 and RMSE by 677.1 versus RF A. Light-rain errors improved on 30 rows, but moderate rain had 2 rows and heavy rain none; these subgroup results are descriptive and cannot support a causal or robust rainfall claim.
- **Regional findings:** Existing work reports station/metro-line demand differences (LINE02 highest, LINE03 second; exits exceed entries by line). No regional weather forecasting dataset, predictions, or performance metrics were found; geographic regional analysis remains future work.
- **Limitations and next steps:** Citywide aggregate target, one chronological test month, sparse rainy cases, and observed prediction-hour weather in weather models limit operational interpretation. Build a defensible station-to-region map, use archived weather forecasts, include more wet periods, and evaluate with rolling-origin tests and direction/region-specific errors.